In [21]:
from openai import OpenAI
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

# OPENAI API key

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()  # busca automáticamente el archivo .env en el directorio actual

api_key = os.getenv("OPENAI_API_KEY")

# Cargar algunos datos de la Demo del Dataset

In [ ]:
dataset = pd.read_csv("../demo_datasets/demo_tortugas.csv", sep=";").head()
print(dataset)

   id             ficha          especie nombre.especie  fecha_orig  \
0   1  1892 - 1977-1990  Caretta caretta   Tortuga Boba  18/04/1990   
1   2  1868 - 1977-1990  Caretta caretta   Tortuga Boba  14/11/1989   
2   3  9537 - 1998-2010  Caretta caretta   Tortuga Boba  02/12/2010   
3   4  9521 - 1998-2010  Caretta caretta   Tortuga Boba  15/11/2010   
4   5  9436 - 1998-2010  Caretta caretta   Tortuga Boba  29/09/2010   

        fecha  anio         mes   estacion               lugar_orig  ...  \
0  18/04/1990  1990       Abril  Primavera         Rambla de castro  ...   
1  14/11/1989  1989   Noviembre      Otoño                      NaN  ...   
2  02/12/2010  2010   Diciembre   Invierno  CANDELARIA - CANDELARIA  ...   
3  15/11/2010  2010   Noviembre      Otoño             Puerto Colón  ...   
4  29/09/2010  2010  Septiembre      Otoño         EL PORIS - ARICO  ...   

                                           fmt_lugar               muni  \
0  Calle Castro, 38611, Granadilla de Abo

## Función y lectura del texto que se usará como prompt

**Texto usado en el siguiente enlace**: en la carpeta *"prompts"* [llm_system_promt.txt](../prompts/llm_system_promt.txt)

In [24]:
# Función para leer el contenido de un archivo prompt:
def load_prompt(file_path):
    with open(file_path, 'r') as file:
        return file.read()

In [41]:
system_content = load_prompt("../prompts/llm_system_promt.txt")

In [ ]:
def extract_data_from_text(text):
    # Load system content and user content
    #system_content = load_prompt('prompts/llm_system_promt.txt')
    user_content = f"Text: {text}"

    # Initialize the OpenAI API client
    client = OpenAI()

    # Make a request to the OpenAI API to generate a chat completion
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_content
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        #model="gpt-3.5-turbo",
        model="gpt-4.1-mini",
        temperature=0.3,
        max_tokens=4096,
        top_p=0.8,
        response_format={ "type": "json_object" }
    )

    # Extract the completion result and token usage information from the response
    completion = chat_completion.choices[0].message.content
    result = json.loads(completion)
    prompt_tokens_used = chat_completion.usage.prompt_tokens
    completion_tokens_used = chat_completion.usage.completion_tokens

    return result, prompt_tokens_used, completion_tokens_used

## Ejemplo con cadenas de texto sueltas.

In [ ]:

text = "Muy delgada baja movilidad, astenia y anorexia. Se diagnostica desnutricion"
text = "8. ¡¡ no coments, qué olor !!!!"
result, prompt_tokens_used, completion_tokens_used = extract_data_from_text(text)

print(f"Result : {result}")
print(f"Prompt Tokens Used : {prompt_tokens_used}")
print(f"Completion Tokens Used : {completion_tokens_used}")

Result : {'id': 8, 'partes_cuerpo': 'indeterminado', 'daño_enfermedad': 'putrefacción', 'causante': 'indeterminado', 'gravedad': 'fatal', 'estado': 'muerto', 'recogido': 'indeterminado', 'observa': 'olor muy intenso'}
Prompt Tokens Used : 945
Completion Tokens Used : 73


## Ejemplo con un DataFrame de prueba

In [43]:
df_sample = pd.DataFrame(['3. Le falta un trozo de caparazón posterior, mordida de tiburón, con rafia y mucho musgo, necrosada toda la parte posterior, separada la cloaca.',
'4. Anzuelo de palangre en el esófago, hemorragia abundante. Se opera para extraerle el anzuelo y nuere en una hora.',
'5. Con percebes, aparentemente bien, recogida en alta mar.',
#'6. Problema en ojo izdo, nariz y aleta delantera izda. Recogida por un pescador.',
#'7. Con nylon en las aletas, corte en la aleta delantera derecha y problemas en los ojos, muy bébil.',
'8. ¡¡ no coments, qué olor !!!!',
#'9. Se la encontraron flotando a la deriva'
'10. Anzuelo clavado. Recogida en Capitanía de Pto. Colón.  Peso:11,800 kgr.',
'11. Llena de algas',
'12. Musgo y percebes en caparazón. Trozo de caparazón mordido.'
],columns=['text'])
df_sample

,text
0,"3. Le falta un trozo de caparazón posterior, m..."
1,"4. Anzuelo de palangre en el esófago, hemorrag..."
2,"5. Con percebes, aparentemente bien, recogida ..."
3,"8. ¡¡ no coments, qué olor !!!!"
4,10. Anzuelo clavado. Recogida en Capitanía de ...
5,11. Llena de algas
6,12. Musgo y percebes en caparazón. Trozo de ca...


In [44]:
## Función para extraer datos de un dataframe
def extract_data_from_df(df): 
    results = []
    total_prompt_tokens_used = 0
    total_completion_tokens_used = 0

    for text in tqdm(df['text'], desc="Processing texts"):
        result, prompt_tokens_used, completion_tokens_used = extract_data_from_text(text)
        results.append(result)

        total_prompt_tokens_used += prompt_tokens_used
        total_completion_tokens_used += completion_tokens_used

    return pd.DataFrame(results)

In [47]:
df_prueba = extract_data_from_df(df_sample)

Processing texts: 100%|██████████| 7/7 [00:14<00:00,  2.13s/it]


In [46]:
## Prueba 1
df_prueba

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,3,caparazón,"falta, necrosis",tiburón,alta,indeterminado,indeterminado,"Presencia de rafia y mucho musgo, parte poster..."
1,4,esófago,hemorragia,anzuelo,alta,muerto,indeterminado,"Operado para extracción de anzuelo, fallece en..."
2,5,indeterminado,parasitacion,percebes,baja,indeterminado,indeterminado,recogida en alta mar
3,8,indeterminado,putrefacción,indeterminado,fatal,muerto,indeterminado,Olor muy intenso
4,10,indeterminado,anzuelo clavado,anzuelo,media,indeterminado,Capitanía de Pto. Colón,"Peso: 11,800 kgr."
5,11,indeterminado,contaminación,algas,baja,indeterminado,indeterminado,Lleno de algas
6,12,caparazón,"parasitacion, mordida","percebes, mordedor indeterminado",media,indeterminado,indeterminado,Musgo presente en caparazón


In [48]:
## Prueba 2
df_prueba

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,3,caparazón,"herida, necrosis","tiburón, rafia",alta,indeterminado,indeterminado,"musgo en la zona afectada, parte posterior nec..."
1,4,esófago,hemorragia,anzuelo,alta,muerto,indeterminado,"Operado para extracción de anzuelo, fallece en..."
2,5,indeterminado,parasitacion,percebes,baja,indeterminado,indeterminado,recogida en alta mar
3,8,indeterminado,putrefacción,indeterminado,fatal,muerto,indeterminado,Olor muy intenso
4,10,indeterminado,anzuelo clavado,anzuelo,media,indeterminado,Capitanía de Pto. Colón,"Peso: 11,800 kgr."
5,11,indeterminado,contaminación,algas,baja,indeterminado,indeterminado,Animal cubierto de algas
6,12,caparazón,"musgo, percebes, mordida",percebes,baja,indeterminado,indeterminado,Trozo de caparazón mordido


# Probamos con los datos de la Demo de los dataset 

Empezamos con un par de líneas

In [50]:
## Probamos con la demo de los data sets, cogemos las columnas id y las observaciones y las unimos para que tengan el mismo formato que el de prueba
dataset_processed =pd.DataFrame(dataset.id.astype(str) + ". " + dataset.observa, columns=['text']).dropna()
dataset_processed

,text
1,2. 3 años en cautividad en agua dulce.
2,"3. Le falta la aleta delantera dcha, caparazón..."
3,"4. Corte en el cuello por enmallamiento, flaca..."
4,"5. aleta delantera dcha necrosada, jaime 15/10"


In [53]:
df_result_dataset = extract_data_from_df(dataset_processed)

Processing texts: 100%|██████████| 4/4 [00:06<00:00,  1.70s/it]


In [ ]:
## Prueba demo datasets 1
df_result_dataset

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,2,indeterminado,indeterminado,indeterminado,indeterminado,indeterminado,indeterminado,3 años en cautividad en agua dulce
1,3,aleta,falta,indeterminado,alta,indeterminado,indeterminado,"Aleta delantera derecha ausente, peso 20 kg."
2,4,cuello,corte,enmallamiento,alta,moribunda,indeterminado,"flaca, débil"
3,5,aleta,necrosis,indeterminado,alta,indeterminado,jaime,"necrosis en aleta delantera derecha, fecha 15/10"


In [54]:
## Prueba demo datasets 2
df_result_dataset

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,2,indeterminado,indeterminado,indeterminado,indeterminado,indeterminado,indeterminado,3 años en cautividad en agua dulce
1,3,aleta,indeterminado,indeterminado,alta,indeterminado,indeterminado,"Aleta faltante, peso 20 kg."
2,4,cuello,corte,enmallamiento,alta,moribunda,indeterminado,"flaca, débil"
3,5,aleta,necrosis,indeterminado,alta,indeterminado,Jaime,"necrosis en aleta delantera derecha, fecha 15/10"


# Hacemos la prueba final con los datos completos de la demo del dataset

In [66]:
dataset_completo = pd.read_csv("../demo_datasets/demo_tortugas.csv", sep=";")
lineas = str(dataset_completo.shape).replace("(", "").replace(")", "").split(",")[0]
columnas = str(dataset_completo.shape).replace("(", "").replace(")", "").split(",")[1]
print(
f"""
La demo del data set tiene:
- {lineas} líneas (sin descartar las NaN)
- {columnas} columnas
""")


La demo del data set tiene:
- 100 líneas (sin descartar las NaN)
-  23 columnas



In [67]:
dataset_completo_processed =pd.DataFrame(dataset_completo.id.astype(str) + ". " + dataset_completo.observa, columns=['text']).dropna()
dataset_completo_processed.head()

,text
1,2. 3 años en cautividad en agua dulce.
2,"3. Le falta la aleta delantera dcha, caparazón..."
3,"4. Corte en el cuello por enmallamiento, flaca..."
4,"5. aleta delantera dcha necrosada, jaime 15/10"
5,6. aleta delantera y trasera dchas con cortes ...


In [70]:
df_result_dataset_completo = extract_data_from_df(dataset_completo_processed)

Processing texts: 100%|██████████| 92/92 [02:42<00:00,  1.76s/it]


In [ ]:
df_result_dataset_completo.head()

,id,partes_cuerpo,daño_enfermedad,causante,gravedad,estado,recogido,observa
0,2,indeterminado,indeterminado,indeterminado,indeterminado,indeterminado,indeterminado,3 años en cautividad en agua dulce
1,3,aleta,falta,indeterminado,alta,indeterminado,indeterminado,"Aleta delantera derecha ausente, peso 20 kg."
2,4,cuello,corte,enmallamiento,alta,moribunda,indeterminado,"flaca, débil"
3,5,aleta,necrosis,indeterminado,alta,indeterminado,indeterminado,"necrosis en aleta delantera derecha, fecha 15/10"


In [75]:
## Lo guardamos como un csv:
ruta_demo_procesada = "../demo_datasets/demo_procesada/" 
if not os.path.exists(ruta_demo_procesada): 
  os.mkdir(ruta_demo_procesada)

df_result_dataset_completo.to_csv(f"{ruta_demo_procesada}demo_tortugas_result.csv", index=False)

# Datasets procesados en [demo_datasets/demo_procesada/](../demo_datasets/demo_procesada/demo_tortugas_result.csv)